In [ ]:
import sys
import torch

print("Python:", sys.version.split()[0])
print("PyTorch:", torch.__version__)
print("GPU available:", torch.cuda.is_available())

if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
    numbers = torch.tensor([2, 3], device="cuda")
    print("GPU test result:", numbers.sum().item())

In [ ]:
!git init /content/catvton-source
!git -C /content/catvton-source fetch --depth 1 https://huggingface.co/spaces/zhengchong/CatVTON 24a6952918a8c6168b89ae32ad61e7e447e3987b
!git -C /content/catvton-source checkout --detach FETCH_HEAD

In [ ]:
!git -C /content/catvton-source rev-parse --short HEAD
!ls /content/catvton-source/model/pipeline.py

In [ ]:
%pip install accelerate diffusers transformers huggingface_hub

In [ ]:
%cd /content/catvton-source

import torch
from model.pipeline import CatVTONPipeline

print("CatVTON import: OK")
print("GPU available:", torch.cuda.is_available())

In [ ]:
import torch
from huggingface_hub import snapshot_download
from model.pipeline import CatVTONPipeline

checkpoint_dir = snapshot_download(
    repo_id="zhengchong/CatVTON",
    allow_patterns=["mix-48k-1024/attention/*"],
)

pipeline = CatVTONPipeline(
    base_ckpt="booksforcharlie/stable-diffusion-inpainting",
    attn_ckpt=checkpoint_dir,
    attn_ckpt_version="mix",
    weight_dtype=torch.float16,
    device="cuda",
    use_tf32=False,
)

print("CatVTON model loaded on", torch.cuda.get_device_name(0))

In [ ]:
from pathlib import Path

from huggingface_hub import hf_hub_download
from IPython.display import display
from PIL import Image, ImageDraw
from utils import resize_and_crop, resize_and_padding

repo = Path("/content/catvton-source")
examples = repo / "resource/demo/example"

person_file = examples / "person/men/Simon_1.png"
garment_file = next(
    path for path in sorted((examples / "condition/upper").iterdir())
    if path.suffix.lower() in {".jpg", ".jpeg", ".png", ".webp"}
)

def download_sample(path):
    return hf_hub_download(
        repo_id="zhengchong/CatVTON",
        repo_type="space",
        revision="24a6952918a8c6168b89ae32ad61e7e447e3987b",
        filename=path.relative_to(repo).as_posix(),
    )

width, height = 384, 512
person = resize_and_crop(
    Image.open(download_sample(person_file)).convert("RGB"), (width, height)
)
garment = resize_and_padding(
    Image.open(download_sample(garment_file)).convert("RGB"), (width, height)
)

mask = Image.new("L", person.size, 0)
ImageDraw.Draw(mask).polygon(
    [(147, 116), (174, 106), (184, 126), (217, 126),
     (232, 106), (248, 118), (258, 164), (241, 190),
     (238, 263), (155, 263), (151, 190), (133, 164)],
    fill=255,
)

red = Image.new("RGB", person.size, "red")
overlay = Image.blend(person, Image.composite(red, person, mask), 0.35)

print("Person, garment, and mask overlay:")
display(person, garment, overlay)

In [ ]:
from time import perf_counter

torch.cuda.synchronize()
start = perf_counter()

result = pipeline(
    image=person,
    condition_image=garment,
    mask=mask,
    width=width,
    height=height,
    num_inference_steps=25,
    guidance_scale=2.5,
    generator=torch.Generator(device="cuda").manual_seed(42),
)[0]

torch.cuda.synchronize()
print(f"Inference time: {perf_counter() - start:.1f} seconds")

output_path = "/content/tryfit_first_preview.png"
result.save(output_path)
print("Saved:", output_path)
display(result)


In [ ]:
person_path = download_sample(person_file)
garment_path = download_sample(garment_file)
test_width, test_height = 576, 768
test_size = (test_width, test_height)

test_person = resize_and_crop(
    Image.open(person_path).convert("RGB"), test_size
)
test_garment = resize_and_padding(
    Image.open(garment_path).convert("RGB"), test_size
)
test_mask = mask.resize(test_size, Image.Resampling.NEAREST)

sharper_result = pipeline(
    image=test_person,
    condition_image=test_garment,
    mask=test_mask,
    width=test_width,
    height=test_height,
    num_inference_steps=25,
    guidance_scale=2.5,
    generator=torch.Generator(device="cuda").manual_seed(42),
)[0]

sharper_result.save("/content/tryfit_preview_576x768.png")
display(sharper_result)

Manual-mask baseline: CatVTON generated an upper-body try-on preview
on Tesla T4 at 576 × 768, with 25 steps, guidance scale 2.5, and seed 42.
The denoising progress bar showed about 16 seconds.
The garment looked clearer than in the 384 × 512 test.
The mask was drawn manually for this sample.

In [ ]:
from importlib.util import find_spec

for package in ("detectron2", "cv2", "scipy"):
    status = "found" if find_spec(package) else "missing"
    print(f"{package}: {status}")

In [ ]:
%pip install av
%pip install --no-build-isolation "git+https://github.com/facebookresearch/detectron2.git"

In [ ]:
import torch
from detectron2 import _C

print("Detectron2 extension: OK")
print("GPU available:", torch.cuda.is_available())

In [ ]:
%cd /content/catvton-source
from pathlib import Path

import torch
from huggingface_hub import snapshot_download
from IPython.display import display
from PIL import Image
from model.cloth_masker import AutoMasker

mask_checkpoints = Path(snapshot_download(
    repo_id="zhengchong/CatVTON",
    allow_patterns=["DensePose/*", "SCHP/*"],
))

automasker = AutoMasker(
    densepose_ckpt=str(mask_checkpoints / "DensePose"),
    schp_ckpt=str(mask_checkpoints / "SCHP"),
    device="cuda",
)

with torch.inference_mode():
    auto_mask = automasker(test_person, mask_type="upper")["mask"].convert("L")

auto_mask = auto_mask.resize(test_person.size, Image.Resampling.NEAREST)
red = Image.new("RGB", test_person.size, "red")
overlay = Image.composite(red, test_person, auto_mask)

print("Original photo and automatic upper-body mask:")
display(test_person, Image.blend(test_person, overlay, 0.35))

In [ ]:
auto_result = pipeline(
    image=test_person,
    condition_image=test_garment,
    mask=auto_mask,
    width=576,
    height=768,
    num_inference_steps=25,
    guidance_scale=2.5,
    generator=torch.Generator(device="cuda").manual_seed(42),
)[0]

display(auto_result)

CatVTON AutoMasker generated an upper-body mask and a 576 × 768
try-on preview on a Tesla T4 without a manually drawn mask.
Settings: 25 steps, guidance scale 2.5, seed 42.
The denoising progress bar showed about 16 seconds for this sample.